# Text-to-SQL: Phase 2 + Phase 10 Setup
**AIML Capstone — Project 6 | Kethan Uppalapati**

Repo: `https://github.com/kethansplunk/Codegen` | Branch: `Colabsetup_modeltrain`

This notebook covers:
- **Phase 2** — Google Colab Environment Setup (Drive mount, GitHub sync, deps install)
- **Phase 10** — Download Qwen3-7B and Qwen2.5-Coder-7B-Instruct from HuggingFace

> ⚠️ **Before running**: Go to `Runtime → Change runtime type → GPU → T4` and click Save.

---
## How to use
1. Run **Cell 1** at the start of **every** Colab session.
2. Run **Cell 2** to verify GPU.
3. Run **Cells 3–5** to create folders and Drive checkpoint directory.
4. Run **Cells 6–9** (Phase 10) once per session to download base models.

---
# PHASE 2 — Google Colab Environment Setup
### Cell 1 — Run at the start of EVERY Colab session

In [ ]:
# ── CELL 1: Run at the start of every Colab session ──────────────────────────

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone or pull your GitHub branch
import os

REPO   = "https://github.com/kethansplunk/Codegen.git"
BRANCH = "Colabsetup_modeltrain"
REPO_DIR = "/content/Codegen"

if not os.path.exists(REPO_DIR):
    print(f"Cloning branch '{BRANCH}' from {REPO} ...")
    os.system(f"git clone --branch {BRANCH} {REPO}")
else:
    print(f"Pulling latest changes from branch '{BRANCH}' ...")
    os.system(f"cd {REPO_DIR} && git fetch && git reset --hard origin/{BRANCH}")

%cd {REPO_DIR}
print(f"\nWorking directory: {os.getcwd()}")
print(f"Branch: {os.popen('git branch --show-current').read().strip()}")

# 3. Install all dependencies
print("\nInstalling dependencies (~3-5 min on first run)...")
!pip install -q transformers==4.45.0 accelerate peft trl
!pip install -q networkx rapidfuzz bm25s chromadb sqlglot
!pip install -q sentence-transformers huggingface_hub
!pip install -q python-dotenv openai pandas numpy tqdm pyyaml rich tiktoken
!python -m spacy download en_core_web_sm -q

print("\n✅ Phase 2 Cell 1 complete — Drive mounted, repo pulled, deps installed.")

### Cell 2 — Verify GPU

In [ ]:
# ── CELL 2: Verify GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"✅ GPU : {gpu_name}")
    print(f"   VRAM: {vram_gb} GB")
    if vram_gb < 16:
        print("⚠️  Less than 16 GB VRAM. SchemaLinker GRPO (Phase 12) will need batch size reduction.")
    elif vram_gb >= 40:
        print("🚀 A100 — ideal for all training phases.")
    else:
        print("👍 T4 16 GB — sufficient for SFT and SAR. GRPO will need gradient checkpointing.")
else:
    print("❌ No GPU. Go to Runtime → Change runtime type → GPU → T4, then re-run.")

### Cell 3 — Create directory structure

In [ ]:
# ── CELL 3: Create directory structure ───────────────────────────────────────
import os

dirs = [
    "data/spider",
    "data/processed",
    "data/fk_graphs",
    "models/qwen3-7b",
    "models/qwen25-coder-7b",
    "models/schema_linker_trained",
    "models/sar_trained",
    "src",
    "indexes/chroma_spider",
    "logs",
    "configs",
    "external",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Directory structure created:")
!find . -maxdepth 2 -type d | sort

### Cell 4 — Create Drive checkpoint folder

In [ ]:
# ── CELL 4: Create Drive checkpoint folder ───────────────────────────────────
import os

DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/text2sql_checkpoints/"
os.makedirs(DRIVE_CHECKPOINT_PATH, exist_ok=True)

print(f"✅ Drive checkpoint folder ready: {DRIVE_CHECKPOINT_PATH}")
print("Contents:")
!ls /content/drive/MyDrive/text2sql_checkpoints/

### Cell 5 — Auto-save config (reference snippet for training phases)

In [ ]:
# ── CELL 5: Auto-save config reference ───────────────────────────────────────
# This snippet goes into your TrainingArguments in Phase 12 and 13.
# Saves a checkpoint every 200 steps so a Colab disconnect doesn't lose your work.

print("Add this to your TrainingArguments in Phases 12 and 13:")
print("""
training_args = TrainingArguments(
    output_dir="models/schema_linker_trained/",  # or models/sar_trained/
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    gradient_checkpointing=True,   # essential for T4 16 GB
    fp16=True,                     # halves VRAM usage on T4
    ...
)
""")
print("⚠️  T4 tip: always set gradient_checkpointing=True and fp16=True to avoid OOM errors.")

---
# PHASE 10 — Download Models

> ⏱️ Each model is ~15 GB. Allow **10–20 minutes per model**.
>
> ⚠️ Models in `/content/` are lost when the session ends. Only trained checkpoints go to Drive.

### Cell 6 — HuggingFace login

In [ ]:
# ── CELL 6: HuggingFace login ─────────────────────────────────────────────────
# Get your token from: https://huggingface.co/settings/tokens
# Role: Read is sufficient

import os
from huggingface_hub import login

HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"  # ← replace this

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ HuggingFace login successful")

### Cell 7 — Download Qwen3-7B (SchemaLinker base)
Used in Phase 12 (SchemaLinker SFT + GRPO). ~15 GB.

In [ ]:
# ── CELL 7: Download Qwen3-7B ─────────────────────────────────────────────────
import os

print("Downloading Qwen/Qwen3-7B (~15 GB) — 10-20 minutes...")
!huggingface-cli download Qwen/Qwen3-7B \
    --local-dir models/qwen3-7b \
    --token $HF_TOKEN

# Verify
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained("models/qwen3-7b")
    print(f"\n✅ Qwen3-7B ready | Vocab size: {tok.vocab_size}")
except Exception as e:
    print(f"\n❌ Load failed: {e}")

### Cell 8 — Download Qwen2.5-Coder-7B-Instruct (SQL generator)
Used in Phase 15 (SQL generation). ~15 GB.

In [ ]:
# ── CELL 8: Download Qwen2.5-Coder-7B-Instruct ───────────────────────────────
import os

print("Downloading Qwen/Qwen2.5-Coder-7B-Instruct (~15 GB) — 10-20 minutes...")
!huggingface-cli download Qwen/Qwen2.5-Coder-7B-Instruct \
    --local-dir models/qwen25-coder-7b \
    --token $HF_TOKEN

# Verify
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained("models/qwen25-coder-7b")
    print(f"\n✅ Qwen2.5-Coder-7B-Instruct ready | Vocab size: {tok.vocab_size}")
except Exception as e:
    print(f"\n❌ Load failed: {e}")

### Cell 9 — Verify both models + GPU memory check

In [ ]:
# ── CELL 9: Verify both models ────────────────────────────────────────────────
import os, torch
from transformers import AutoTokenizer

models_to_check = {
    "Qwen3-7B"                 : "models/qwen3-7b",
    "Qwen2.5-Coder-7B-Instruct": "models/qwen25-coder-7b",
}

print("=" * 50)
print("MODEL VERIFICATION")
print("=" * 50)

all_ok = True
for name, path in models_to_check.items():
    if not os.path.exists(path):
        print(f"❌ {name}: not found at {path}")
        all_ok = False
        continue
    size = os.popen(f"du -sh {path} 2>/dev/null").read().split()[0]
    try:
        tok = AutoTokenizer.from_pretrained(path)
        print(f"✅ {name}")
        print(f"   Path      : {path}")
        print(f"   Disk size : {size}")
        print(f"   Vocab size: {tok.vocab_size}\n")
    except Exception as e:
        print(f"❌ {name}: tokenizer failed — {e}\n")
        all_ok = False

# GPU memory
if torch.cuda.is_available():
    free  = round(torch.cuda.mem_get_info()[0] / 1e9, 1)
    total = round(torch.cuda.mem_get_info()[1] / 1e9, 1)
    print(f"GPU memory: {free} GB free / {total} GB total")

print()
if all_ok:
    print("🎉 Phase 10 complete — both models ready.")
    print("   Next: Phase 11 (CoT data generation on Mac) → Phase 12 (SchemaLinker SFT on Colab).")
else:
    print("⚠️  Some models missing. Re-run Cell 7 and/or Cell 8.")

---
# Utility — Save Checkpoint to Drive
Run after Phase 12 (schema_linker_trained) or Phase 13 (sar_trained) completes.

In [ ]:
# ── Save checkpoint to Google Drive ──────────────────────────────────────────
import shutil, os

# Change per phase: "schema_linker_trained" or "sar_trained"
CHECKPOINT_NAME = "schema_linker_trained"

DRIVE_PATH = "/content/drive/MyDrive/text2sql_checkpoints/"
src = f"models/{CHECKPOINT_NAME}/"
dst = f"{DRIVE_PATH}{CHECKPOINT_NAME}/"

if not os.path.exists(src):
    print(f"❌ Not found: {src} — did training complete?")
else:
    os.makedirs(DRIVE_PATH, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    size = os.popen(f"du -sh {dst}").read().split()[0]
    print(f"✅ Saved to Drive: {dst} ({size})")

# Utility — Restore Checkpoint from Drive
Use at the start of a new session to restore a previously saved checkpoint.

In [ ]:
# ── Restore checkpoint from Drive ────────────────────────────────────────────
import shutil, os

CHECKPOINT_NAME = "schema_linker_trained"  # or "sar_trained"

DRIVE_PATH = "/content/drive/MyDrive/text2sql_checkpoints/"
src = f"{DRIVE_PATH}{CHECKPOINT_NAME}/"
dst = f"models/{CHECKPOINT_NAME}/"

if not os.path.exists(src):
    print(f"❌ Not found in Drive: {src}")
    print("Available checkpoints:")
    !ls /content/drive/MyDrive/text2sql_checkpoints/ 2>/dev/null
else:
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"✅ Restored to {dst}")
    !ls {dst}